# Design Considerations

## Identification of Five Design Considerations (Module 8)

### Design Decision 1: Pre-Processing Techniques

**Significance:** Preprocessing is crucial for ensuring that the input images are in a consistent format, which improves the performance and reliability of the model. Different preprocessing techniques can affect the quality of the embeddings and the overall accuracy of the system.

**Plan:**
Compare different pre-processing techniques such as resizing, normalization, histogram equalization, and color adjustments. Measure the impact of the techniques by evaluating the impact on model accuracy and embedding quality. Plot the preprocessing steps and their effect on sample images to visualize the results.

### Design Decision 2: Embedding Dimensionality

**Significance:** The dimensionality of embeddings affects both the accuracy and the computational efficiency of the system. Higher dimensions may capture more information but can also lead to overfitting and increased computational cost.

**Plan:** Experiment with differnet dimensions and test embedding sizes, such as 128, 256, 512. Evaluate the performance by measuring the accuracy, precision, recall and computational time for each dimensionality. Plot the results to visualize the performance metrics for comparison.

### Design Decision 3: Distance Metrics for Nearest Neighbor Search

**Significance:** The choice of distance metric impacts the accuracy and efficiency of the nearest neighbor search, which is critical for the visual search system's performance.

**Plan:** Evaluate and compare different distance metrics such as the Euclidean, Cosine and Manhattan distances. Evaluate the impact of the distance metrics by analyzing the accuracy, precision, recall and computation time. Plot the results to visualize the performance metrics for comparison.

### Design Decision 4: Data Augmentation Strategies

**Significance:** Data augmentation increases the diversity of the training dataset, which helps in improving the generalization of the model and handling variations in input images.

**Plan:** Implement various data augmentation strategies such as rotation, flipping, zoom, and color jitter to the gallery images. Evaluate the impact by measuring the accuracy, precision, and recall of the model trained with augmented data. Visualize and compare the performance metrics with and without augmentation.

### Design Decision 5: Handling Class Imbalance

**Significance:** Class imbalance occurs when the number of images per person varies significantly, which can affect the model's performance. Models trained on imbalanced datasets tend to favor the majority classes, leading to poor performance on minority classes.

**Plan:** Analyze the distribution of images per person to identify the extent of class imbalance. Implement techniques like oversampling and synthetic data generation. Evaluate the impact by measuring the performance of the model with and without class balancing techniques. In the process, it is also possible to fine-tune for the optimal number of images per person for the optimal results. Visualize and compare performance metrics to determine the effectiveness of each technique. 

## Analysis of Two Design Considerations (Module 8)

### Data Augmentation

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from albumentations import Compose, HorizontalFlip, RandomBrightnessContrast, Rotate
import cv2

# Example data augmentation pipeline
augmentations = Compose([
    HorizontalFlip(p=0.5),
    RandomBrightnessContrast(p=0.5),
    Rotate(limit=15, p=0.5)
])

def augment_image(image):
    augmented = augmentations(image=image)
    return augmented['image']

# Load and augment a sample image
sample_image_path = 'storage/gallery/Aaron_Sorkin/Aaron_Sorkin_0001.jpg'
sample_image = cv2.imread(sample_image_path)
augmented_image = augment_image(sample_image)

# Display original and augmented images
plt.figure(figsize=(10, 5))
plt.subplot(1, 2, 1)
plt.title('Original Image')
plt.imshow(cv2.cvtColor(sample_image, cv2.COLOR_BGR2RGB))

plt.subplot(1, 2, 2)
plt.title('Augmented Image')
plt.imshow(cv2.cvtColor(augmented_image, cv2.COLOR_BGR2RGB))
plt.show()

Data augmentation can significantly enhance the performance and robustness of the visual search system. Introducing variability for each person in the storage gallery may help the model generalize better to the probe input image. By applying transformations such as horizontal flipping, random brightness and contrast adjustments, rotation, scaling, translation, and adding Gaussian noise, the model is exposed to a broader range of scenarios and conditions, and can help augment and adapt to different real-world conditions during probe detection. 

### Class Imbalance

In [ ]:
import os
import numpy as np
from PIL import Image, ImageEnhance, ImageOps
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from imblearn.over_sampling import SMOTE
from sklearn.utils import resample

# Paths to the storage directories
GALLERY_STORAGE = "storage/gallery"
EMBEDDINGS_STORAGE = "storage/embeddings"

# Create output directory for plots and analysis results
output_dir = 'visual_search_system/analysis/plots'
os.makedirs(output_dir, exist_ok=True)

# Function to load all images from the gallery
def load_images_from_gallery(gallery_path):
    images = []
    for folder in os.listdir(gallery_path):
        folder_path = os.path.join(gallery_path, folder)
        if not os.path.isdir(folder_path):  # Skip non-directory files
            continue
        for file in os.listdir(folder_path):
            if file.endswith(('.jpg', '.jpeg', '.png')):
                file_path = os.path.join(folder_path, file)
                images.append((folder, file, Image.open(file_path).convert('RGB')))
    return images

# Function to preprocess images
def preprocess_image(image):
    image = image.resize((224, 224))
    return np.array(image) / 255.0

# Function to augment an image
def augment_image(image):
    augmentations = [
        ImageOps.mirror,
        ImageOps.flip,
        lambda img: img.rotate(90),
        lambda img: img.rotate(180),
        lambda img: img.rotate(270),
        lambda img: ImageEnhance.Contrast(img).enhance(1.5),
        lambda img: ImageEnhance.Brightness(img).enhance(1.5)
    ]
    augmented_images = [augmentation(image) for augmentation in augmentations]
    return augmented_images

# Function to balance the dataset using oversampling
def balance_dataset(X, y):
    # Encode string labels to integers
    label_encoder = LabelEncoder()
    y_encoded = label_encoder.fit_transform(y)
    
    # Separate the majority and minority classes
    class_counts = np.bincount(y_encoded)
    minority_classes = np.where(class_counts < 6)[0]
    majority_classes = np.where(class_counts >= 6)[0]
    
    X_balanced = []
    y_balanced = []
    
    # Oversample minority classes
    for label in minority_classes:
        X_class = X[y_encoded == label]
        while len(X_class) < 6:
            augmented_images = [augment_image(Image.fromarray((img * 255).astype(np.uint8))) for img in X_class]
            augmented_images = np.vstack(augmented_images)
            X_class = np.vstack((X_class, augmented_images[:6 - len(X_class)]))
        X_balanced.append(X_class)
        y_balanced.extend([label] * len(X_class))
    
    # Add majority classes
    for label in majority_classes:
        X_class = X[y_encoded == label]
        X_balanced.append(X_class)
        y_balanced.extend([label] * len(X_class))
    
    X_balanced = np.vstack(X_balanced)
    y_balanced = np.array(y_balanced)
    
    # Decode the integer labels back to original string labels
    y_balanced = label_encoder.inverse_transform(y_balanced)
    
    return X_balanced, y_balanced


In [ ]:
# Load and preprocess images
images = load_images_from_gallery(GALLERY_STORAGE)
X = np.array([preprocess_image(img) for _, _, img in images])
y = np.array([folder for folder, _, _ in images])

# Plot the initial distribution of image counts per person
unique, counts = np.unique(y, return_counts=True)
class_distribution_initial = dict(zip(unique, counts))

plt.figure(figsize=(12, 6))
sns.histplot(list(class_distribution_initial.values()), bins=10, kde=True)
plt.title('Initial Distribution of Image Counts per Person')
plt.xlabel('Number of Images')
plt.ylabel('Frequency')
plt.savefig(os.path.join(output_dir, 'initial_image_counts_distribution.png'))
plt.show()

In [ ]:
# Balance the dataset
X_balanced, y_balanced = balance_dataset(X, y)

# Check the new class distribution
unique, counts = np.unique(y_balanced, return_counts=True)
class_distribution = dict(zip(unique, counts))

# Plot the new distribution
plt.figure(figsize=(12, 6))
sns.histplot(list(class_distribution.values()), bins=10, kde=True)
plt.title('Distribution of Image Counts per Person after Balancing')
plt.xlabel('Number of Images')
plt.ylabel('Frequency')
plt.savefig(os.path.join(output_dir, 'balanced_image_counts_distribution.png'))
plt.show()

Understanding class distribution is essential in visual search systems to identify and address class imbalances that can bias model performance. A skewed class distribution can lead to models that perform well on majority classes but poorly on minority ones, resulting in unfair and unreliable outcomes. By analyzing and balancing the class distribution through techniques like oversampling and augmentation, we can ensure that the model is trained on a more representative dataset. This leads to improved generalization, fairer evaluations, and more robust performance metrics, making the model more applicable and trustworthy in real-world scenarios.

## Analysis of System Parameters and Configurations (Module 9)

### Design Decision 1: Choice of Model Architecture

**Significance:** The choice of model architecture, such as ResNet-18 or ResNet-34, directly affects the system's accuracy and computational efficiency. ResNet-18 offers a simpler model with fewer layers, which can be beneficial for faster inference times and lower computational costs, while ResNet-34 provides deeper feature extraction capabilities at the cost of increased computation.

**Plan:** We can compare the performance of ResNet-18 and ResNet-34 in terms of accuracy, inference time, and resource usage. Use a validation sample dataset to evaluate precision, recall, and system latency for both models.

**Implemented Design:** The code supports the selection of ResNet-18 and ResNet-34 models, trained using the SimCLR framework for self-supervised learning. The models are loaded based on user input for image size and ResNet variant.

### Design Decision 2: Image Preprocessing

**Significance:** Preprocessing the images by resizing them ensures consistency in input dimensions, which is crucial for the deep learning model to perform accurate feature extraction. It also helps in normalizing the images under different lighting conditions, enhancing model robustness. Additionally, the image size impacts model latency, with smaller images decreasing the computational load and processing time but potentially lower accuracy due to loss of detail. The two image size options are 64x64 and 224x224. Between the two options, 224x224 is a little more than 12 times larger than 64x64. Choosing a smaller image size involves a trade-off between speed and accuracy. 

**Plan:** We can evaluate the impact of different image sizes (e.g., 64x64 and 224x224) on model latency and accuracy. Measure the inference time and accuracy for each image size to find an optimal balance between processing speed and identification accuracy. We can also evaluate the impact of different image preprocessing techniques (resizing, normalization, and augmentation) on model performance. Analyze the model's accuracy and robustness under various lighting conditions and angles.

**Implemented Design:** Images are resized to a consistent size during preprocessing, ensuring uniformity and facilitating efficient processing as input for the deep learning model. The model is loaded based on user input for the image size, and the images are pre-processed accordingly. 

### Design Decision 3: Dynamic Access Control Management

**Significance:** The ability to dynamically add and remove personnel ensures that the system remains up-to-date with the latest personnel changes. This is critical for maintaining security and operational efficiency.

**Plan:** Test the system's ability to handle real-time updates to the personnel database. Measure the time taken to add and remove personnel and the impact on the KDTree's performance.

**Implemented Design:** The system includes methods to add new personnel by generating and storing new embeddings and to remove departing personnel by deleting their embeddings and associated images. Deleting the embeddings ensures that the model is not accidentally initiated with removed individuals. Adding or re-adding the personnel would require human intervention to securely validate the employee before introducing it to the system.

### Design Decision 4: Embedding Storage and Search with KDTree

**Significance:** Efficient storage and retrieval of embeddings are crucial for the real-time identification of personnel. Using KDTree for nearest neighbor search ensures fast and accurate retrieval of high-dimensional embeddings, essential for scaling the system to handle millions of personnel. 

KDTrees enable efficient searches in high-dimensional spaces by recursively partitioning the data into k-dimensional subspaces. This reduces the time complexity of nearest neighbor searches from linear to logarithmic time in practice, significantly speeding up the identification process. KDTree structures can handle large datasets efficiently, making them suitable for scaling the system to support millions of personnel. The hierarchical structure of KDTrees allows them to manage high-dimensional data effectively. By organizing data points in a way that minimizes the number of distance calculations needed to find the nearest neighbors, KDTrees maintain high accuracy in identifying the closest embeddings, crucial for reliable personnel identification. KDTrees can also be used with various distance metrics (e.g., Euclidean, Manhattan), providing flexibility in adapting the search to different types of data distributions and requirements.

**Plan:** Measure the retrieval speed and accuracy of the KDTree implementation. Compare it with other indexing methods (e.g., Ball Tree, FLANN) to ensure optimal performance for large-scale deployment.

**Implemented Design:** Embeddings are stored in a KDTree, allowing fast retrieval of nearest neighbors in the high-dimensional embedding space.

### Design Decision 5: Use of SimCLR for Self-Supervised Learning

**Significance:** The use of SimCLR (Simple Framework for Contrastive Learning of Visual Representations) for self-supervised learning is a crucial design decision. SimCLR enables the model to learn robust and generalizable features without requiring labeled data, which can be expensive and time-consuming to obtain. By leveraging contrastive learning, SimCLR trains the model to distinguish between different images by maximizing agreement between differently augmented views of the same image. This approach enhances the model's ability to perform accurate feature extraction under various conditions.

**Plan:** We can analyze the impact of using SimCLR on the model's performance compared to traditional supervised learning methods. Evaluate the model's accuracy, robustness, and generalization ability on different datasets. Assess the improvement in training efficiency and reduction in labeling costs.

**Implemented Design:** The deep learning model is trained using the SimCLR framework, which involves augmenting images and training the model to recognize that these different versions of the same image are related.